# Day 7: Tools - Letting the LLM Use Your Code

**LangChain Hands-on Series | Day 7 of 10**
Trainer: Ajeetkumar

---

Ask an LLM "what is 48,213 x 9,871?" and it might get it slightly wrong. Ask "what is the status of order ORD-1002?" and it cannot know - that lives in your database. LLMs are great with language but they cannot calculate reliably, they cannot see live data, and they cannot *do* anything.

**Tools** fix this. A tool is just a Python function that you describe to the model. The model reads the description and, when it needs it, says: *"please run `lookup_order` with `order_id='ORD-1002'`"*. Your code runs the function and gives the result back.

The most important thing to understand today:

> **The model never runs your code.** It only *asks* for a tool call. Your program decides whether to run it.

It is like a manager and an assistant. The manager (LLM) says "get me the sales report for March". The assistant (your code) actually opens the system, gets the report, and hands it back. Then the manager writes the final answer.

```
  user question
       |
       v
  +---------+   "call calculator(expression='48213*9871')"   +-------------------+
  |   LLM   | ----------------------------------------------> |  YOUR CODE runs   |
  |         | <---------------------------------------------- |  the function     |
  +---------+        ToolMessage("475910523")                 +-------------------+
       |
       v
  "48,213 x 9,871 = 475,910,523"
```

Today's plan:

1. Create tools with the `@tool` decorator
2. Give the model tools with `bind_tools` and read its tool requests
3. Run the tools and send results back - the full loop
4. Several tools, forcing a tool, handling errors
5. Build a reusable tool loop (a mini agent - the bridge to Day 8)
6. **Mini project:** a CloudDesk customer support assistant with real data lookups

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
print("Ready. Model:", MODEL_NAME)

## 1. Creating a tool

Put `@tool` above a normal function. Three things become the tool's "instruction manual" for the model, so write them carefully:

- the **function name** - becomes the tool name
- the **docstring** - tells the model *when* to use it
- the **type hints** - tell the model what arguments to send

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers. Use this for any multiplication."""
    return a * b

print("Name       :", multiply.name)
print("Description:", multiply.description)
print("Arguments  :", multiply.args)

A tool is a Runnable (yes, again), so you can test it directly. Always test your tools on their own before giving them to a model - it makes debugging much easier.

In [ ]:
multiply.invoke({"a": 6, "b": 7})

### Better argument descriptions with Pydantic

For anything beyond trivial arguments, describe each one. The model reads these descriptions to decide what to fill in.

In [ ]:
from pydantic import BaseModel, Field

class CurrencyInput(BaseModel):
    amount: float = Field(description="The amount of money to convert")
    from_currency: str = Field(description="3-letter currency code to convert from, e.g. USD")
    to_currency: str = Field(description="3-letter currency code to convert to, e.g. INR")

RATES_TO_INR = {"INR": 1.0, "USD": 88.0, "EUR": 96.0, "GBP": 112.0}     # fixed demo rates, not live

@tool(args_schema=CurrencyInput)
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between currencies (INR, USD, EUR, GBP) using fixed demo exchange rates."""
    for code in (from_currency, to_currency):
        if code.upper() not in RATES_TO_INR:                 # unknown code: explain instead of crashing
            return f"Sorry, I only know these currencies: {', '.join(RATES_TO_INR)}."
    inr = amount * RATES_TO_INR[from_currency.upper()]
    result = inr / RATES_TO_INR[to_currency.upper()]
    return f"{amount} {from_currency.upper()} = {result:,.2f} {to_currency.upper()}"

convert_currency.invoke({"amount": 100, "from_currency": "USD", "to_currency": "INR"})

Two more tools we will use today. The calculator only allows digits and maths symbols before evaluating - never run `eval` on raw text that came from a model or a user.

In [ ]:
import re

@tool
def calculator(expression: str) -> str:
    """Evaluate a maths expression such as '48213 * 9871' or '(120 + 30) / 4'. Use for any arithmetic."""
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression):     # only allow safe characters
        return "Error: only numbers and + - * / ( ) % are allowed."
    return str(eval(expression))

WEATHER = {"pune": "29 C, partly cloudy", "mumbai": "31 C, humid", "delhi": "34 C, sunny", "bengaluru": "24 C, light rain"}

@tool
def get_weather(city: str) -> str:
    """Get the current weather for an Indian city (demo data)."""
    return WEATHER.get(city.lower(), f"No weather data for {city}.")

print(calculator.invoke({"expression": "48213 * 9871"}))
print(get_weather.invoke({"city": "Pune"}))

## 2. Giving tools to the model

`bind_tools` attaches the tool descriptions to every request. The model now *knows* the tools exist.

In [ ]:
tools = [calculator, get_weather, convert_currency]
llm_with_tools = llm.bind_tools(tools)

**Pause and predict:** we ask "What is 48213 times 9871?". Will `response.content` contain the number?

In [ ]:
response = llm_with_tools.invoke("What is 48213 times 9871?")

print("content   :", repr(response.content))
print("tool_calls:", response.tool_calls)

<details>
<summary>Click to see the explanation</summary>

No. `content` is empty. Instead, the model filled `tool_calls` with a request: run `calculator` with an `expression`. The model did not calculate anything - it asked us to. Nothing has actually run yet.
</details>

Let's look at one tool call closely:

```
{
  "name": "calculator",                      <- which tool
  "args": {"expression": "48213 * 9871"},    <- with what arguments
  "id":   "call_abc123"                      <- an id, so the result can be matched to this request
}
```

**Not every question needs a tool.** When the model can answer by itself, it just answers:

In [ ]:
plain = llm_with_tools.invoke("Say hello in French.")
print("content   :", plain.content)
print("tool_calls:", plain.tool_calls)

## 3. Completing the loop

Now we play the "assistant" role: run the requested tool and send the result back as a `ToolMessage`. Then the model writes the final answer.

```
 messages = [Human]
     -> LLM ->  AIMessage(tool_calls=[calculator(...)])     add to messages
     -> we run calculator  ->  ToolMessage("475910523")     add to messages
     -> LLM ->  AIMessage("The answer is 475,910,523")      final answer
```

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="What is 48213 times 9871?")]

ai_msg = llm_with_tools.invoke(messages)        # step 1: the model asks for a tool
messages.append(ai_msg)

ai_msg.tool_calls

In [ ]:
tools_by_name = {t.name: t for t in tools}       # look-up table: name -> tool object

for call in ai_msg.tool_calls:
    selected_tool = tools_by_name[call["name"]]
    tool_message = selected_tool.invoke(call)     # passing the whole call returns a ready-made ToolMessage
    messages.append(tool_message)

tool_message

In [ ]:
final = llm_with_tools.invoke(messages)          # step 3: the model sees the tool result and answers
print(final.content)

Let's print the whole conversation. `pretty_print()` shows each message in a readable format - you can see the full request -> tool -> answer flow.

In [ ]:
for m in messages + [final]:
    m.pretty_print()

## 4. Several tools, and controlling the choice

With several tools bound, the model picks the right one based on the descriptions. Let's check that it chooses sensibly.

In [ ]:
test_questions = [
    "What's the weather like in Mumbai?",
    "Convert 250 USD to INR",
    "What is (1500 + 2750) / 5?",
    "Who wrote the Ramayana?",
]

for q in test_questions:
    calls = llm_with_tools.invoke(q).tool_calls
    chosen = calls[0]["name"] if calls else "(no tool - answered directly)"
    print(f"{q:<38} -> {chosen}")

### Forcing a tool with `tool_choice`

Sometimes you want to *guarantee* a tool call - for example, a form-filling step where the model must use `convert_currency`.

| `tool_choice` | Meaning |
|---------------|---------|
| `"auto"` (default) | model decides |
| `"any"` | must call some tool |
| `"get_weather"` | must call that specific tool |

In [ ]:
forced = llm.bind_tools(tools, tool_choice="get_weather")
forced.invoke("I'm travelling to Delhi tomorrow.").tool_calls

### When a tool fails

Real tools fail: an API is down, an ID does not exist. If a tool raises an exception, the whole program crashes. A better approach is to turn the error into a message so the model can tell the user what went wrong - or try again with better arguments.

In [ ]:
from langchain_core.tools import ToolException

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ToolException("Cannot divide by zero. Ask the user for a different number.")
    return a / b

divide.handle_tool_error = True        # turn ToolException into a normal text result instead of crashing

print(divide.invoke({"a": 10, "b": 2}))
print(divide.invoke({"a": 10, "b": 0}))

## 5. A reusable tool loop - your first mini agent

Sometimes one tool call is not enough. "Convert 100 USD to INR and then multiply by 12" needs two calls, one after the other. So we repeat:

```
          +-------------------------------------------+
          |                                           |
          v                                           |
   call the LLM  --> tool calls? --yes--> run them, add ToolMessages
                          |
                          no
                          |
                          v
                   final answer, stop
```

This loop is the heart of every agent. Let's write it ourselves once, so tomorrow's `create_agent` feels obvious instead of magical.

In [ ]:
from langchain_core.messages import SystemMessage

def run_with_tools(question, tools, max_steps=5, verbose=True):
    model = llm.bind_tools(tools)
    lookup = {t.name: t for t in tools}
    messages = [SystemMessage(content="Use the tools when helpful. Be concise."),
                HumanMessage(content=question)]

    for step in range(1, max_steps + 1):            # safety limit so it can never loop forever
        ai = model.invoke(messages)
        messages.append(ai)

        if not ai.tool_calls:                       # no more tools needed -> we are done
            return ai.content, messages

        for call in ai.tool_calls:
            result = lookup[call["name"]].invoke(call)
            messages.append(result)
            if verbose:
                print(f"step {step}: {call['name']}({call['args']}) -> {result.content}")

    return "Stopped: too many steps.", messages

In [ ]:
answer, trace = run_with_tools("Convert 100 USD to INR, then multiply the result by 12.", tools)
print("\nFinal answer:", answer)

In [ ]:
answer, trace = run_with_tools("What's the weather in Bengaluru and in Pune?", tools)
print("\nFinal answer:", answer)

### Visualization: which tools got used?

Let's run a batch of questions through our loop and count tool usage. In a real product, this kind of chart tells you which tools matter and which nobody uses.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

questions = [
    "Weather in Delhi?", "Convert 50 EUR to INR", "What is 15% of 2400? Use the calculator with 2400*15/100",
    "Weather in Mumbai?", "Convert 1000 INR to USD", "What is 999 * 3?", "Tell me a joke",
]

usage = Counter()
for q in questions:
    _, msgs = run_with_tools(q, tools, verbose=False)
    for m in msgs:
        for call in getattr(m, "tool_calls", []) or []:
            usage[call["name"]] += 1

plt.figure(figsize=(5, 3))
plt.bar(usage.keys(), usage.values(), color="#4C72B0")
plt.title("Tool calls across 7 questions")
plt.ylabel("Times called")
plt.show()

## Mini project: CloudDesk support assistant

Let's give the model access to real (well, demo) company data: the orders in `data/orders.csv`. The assistant should be able to look up an order, check the refund policy, and calculate a pro-rata refund.

In [ ]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
orders

In [ ]:
@tool
def lookup_order(order_id: str) -> str:
    """Look up a CloudDesk order by its id (format ORD-1234). Returns customer, plan, status and amount."""
    row = orders[orders["order_id"].str.upper() == order_id.upper()]
    if row.empty:
        return f"No order found with id {order_id}."
    return row.iloc[0].to_json()                       # the whole row as JSON text

@tool
def refund_policy() -> str:
    """Return CloudDesk's refund policy. Use when the user asks about refunds or cancellations."""
    return ("Annual subscriptions can be refunded pro-rata within the first 30 days. "
            "Monthly subscriptions are not refundable but can be cancelled at any time.")

@tool
def prorata_refund(amount_paid: float, days_used: int, total_days: int = 365) -> str:
    """Calculate a pro-rata refund: the unused share of the amount paid."""
    refund = amount_paid * (total_days - days_used) / total_days
    return f"Refund: USD {refund:,.2f}"

support_tools = [lookup_order, refund_policy, prorata_refund]

print(lookup_order.invoke({"order_id": "ORD-1002"}))

In [ ]:
answer, _ = run_with_tools("What is the status of order ORD-1002 and who is the customer?", support_tools)
print("\nAnswer:", answer)

In [ ]:
answer, _ = run_with_tools("What is your refund policy?", support_tools)
print("\nAnswer:", answer)

In [ ]:
answer, _ = run_with_tools(
    "Order ORD-1003 is an annual plan that was used for 20 days. How much refund would the customer get?",
    support_tools,
)
print("\nAnswer:", answer)

### Your turn

Write a new tool `count_orders_by_status(status: str)` that returns how many orders have a given status (use `orders["status"]`). Add it to `support_tools` and ask: "How many orders are Active?"

In [ ]:
# Write your tool here

## Common mistakes on Day 7

| What you see | What usually went wrong |
|--------------|-------------------------|
| The model never calls your tool | The docstring is vague. Say clearly *when* to use it. |
| Wrong arguments are sent | Add `Field(description=...)` for each argument with examples |
| `400` error about tool messages | Every tool call needs a matching `ToolMessage` before the next model call |
| The program crashes inside a tool | Raise `ToolException` and set `handle_tool_error = True` |
| Endless tool calling | Always have a `max_steps` limit in your loop |

## Quick recap

- `@tool` turns a function into a tool; name, docstring and type hints are its manual.
- `bind_tools` tells the model about tools; the reply contains `tool_calls`, not results.
- Your code runs the tool and sends back a `ToolMessage`; then the model answers.
- Repeat until there are no more tool calls - that loop is an agent.

## Practice before Day 8

1. Add a `get_time(city)` tool and ask a question that needs both weather and time.
2. Add JPY and AED to `RATES_TO_INR` and test a conversion between them.
3. Print the total tokens used by one `run_with_tools` call (hint: sum `usage_metadata` of the AI messages).
4. Ask a question that needs three tool calls in a row and watch the loop.

**Tomorrow:** we replace our hand-written loop with LangChain's `create_agent`, and add memory, streaming, structured answers, human approval and more.